# Designing Reliable Agentic Systems

**Level:** Advanced · **Time:** 90 min

In this comprehensive notebook, we simulate the "Progressive Autonomy Ladder" and application-layer safety controls.

We will cover 4 distinct patterns:
1. **The Deterministic Baseline:** Solving a problem with pure Python code (0% hallucination risk).
2. **The Bounded Agent Upgrade:** Using an LLM to route an edge case the deterministic code couldn't handle.
3. **The Unreliable Swarm:** Demonstrating the latency and cost disaster of throwing 5 agents at a simple problem.
4. **The Idempotent Tool:** Wrapping a dangerous tool with idempotency keys so if the agent loops, the system is safe.

---
## Pattern 1: The Deterministic Baseline (Tier 1)

Always start here. If the input is structured, use code. It is infinitely faster and cheaper than an LLM.

In [1]:
import time

def process_refund_structured(payload: dict):
    start = time.time()
    
    # Deterministic logic
    if payload.get("status") == "failed" and payload.get("amount") < 100:
        decision = "AUTO_REFUND"
    else:
        decision = "MANUAL_REVIEW"
        
    latency = (time.time() - start) * 1000
    print(f"[Deterministic Script] Decision: {decision}. Latency: {latency:.2f}ms. Cost: $0.00")

process_refund_structured({"status": "failed", "amount": 50})


[Deterministic Script] Decision: AUTO_REFUND. Latency: 0.00ms. Cost: $0.00


---
## Pattern 2: The Bounded Agent Upgrade (Tier 2)

If the input is an angry, unstructured customer email, the deterministic script fails. We *promote* the architecture to a single, bounded agent just to parse the intent.

In [2]:
def bounded_parsing_agent(email_text: str):
    start = time.time()
    print("[Agent] Reading unstructured text and mapping to structured JSON...")
    
    # Simulate LLM call
    time.sleep(1.2) 
    
    # The agent outputs structured data to feed back into the Tier 1 deterministic system
    extracted_data = {"status": "failed", "amount": 50}
    
    latency = time.time() - start
    print(f"[Agent] Extracted Data: {extracted_data}. Latency: {latency:.2f}s. Cost: $0.02")
    return extracted_data

email = "I am FURIOUS! You charged me fifty bucks and the app crashed!"
parsed = bounded_parsing_agent(email)
process_refund_structured(parsed)


[Agent] Reading unstructured text and mapping to structured JSON...


[Agent] Extracted Data: {'status': 'failed', 'amount': 50}. Latency: 1.21s. Cost: $0.02
[Deterministic Script] Decision: AUTO_REFUND. Latency: 0.00ms. Cost: $0.00


---
## Pattern 3: The Unreliable Swarm (Anti-Pattern)

What happens if we jump straight to Tier 4 (Multi-Agent Swarm) for this simple problem? Latency skyrockets and costs explode due to coordination overhead.

In [3]:
def multi_agent_overkill(email_text: str):
    start = time.time()
    print("[Manager Agent] I will spawn a team to analyze this.")
    time.sleep(1.0)
    
    print("[Sentiment Agent] Analyzing emotion... it is FURIOUS.")
    time.sleep(1.0)
    
    print("[Financial Agent] Extracting monetary value... $50.")
    time.sleep(1.0)
    
    print("[Reviewer Agent] Validating extraction... looks good.")
    time.sleep(1.0)
    
    print("[Manager Agent] Synthesizing final report...")
    time.sleep(1.0)
    
    latency = time.time() - start
    print(f"[System] Decision Reached. Latency: {latency:.2f}s. Cost: $0.15")

multi_agent_overkill("I am FURIOUS! You charged me fifty bucks and the app crashed!")
print("🚨 CONCLUSION: The Swarm was 5x slower and 7x more expensive than the Bounded Agent for the exact same result.")


[Manager Agent] I will spawn a team to analyze this.


[Sentiment Agent] Analyzing emotion... it is FURIOUS.


[Financial Agent] Extracting monetary value... $50.


[Reviewer Agent] Validating extraction... looks good.


[Manager Agent] Synthesizing final report...


[System] Decision Reached. Latency: 5.01s. Cost: $0.15
🚨 CONCLUSION: The Swarm was 5x slower and 7x more expensive than the Bounded Agent for the exact same result.


---
## Pattern 4: The Idempotent Tool

Agents will loop and retry if they encounter timeouts. You MUST protect destructive actions (like refunds) at the application layer using Idempotency Keys.

In [4]:
# Database of processed transactions
processed_keys = set()

def safe_refund_tool(ticket_id: str, amount: int):
    print(f"\n[Tool Wrapper] Agent attempting refund of ${amount} for ticket {ticket_id}")
    
    # Generate Idempotency Key
    idempotency_key = f"refund_{ticket_id}"
    
    # Application Layer Safety Check
    if idempotency_key in processed_keys:
        print(f"✅ [Safety Layer] BLOCKED: Refund for {ticket_id} already processed. Preventing double-charge.")
        return "ERROR: Refund already processed."
        
    # Process Refund
    processed_keys.add(idempotency_key)
    print(f"💰 [Bank API] Successfully refunded ${amount}.")
    return "SUCCESS"

print("--- The Agent Retries on a Network Timeout ---")
print("[Agent] Calling refund tool (Attempt 1)...")
safe_refund_tool("TKT-999", 50)

print("[Agent] The server timed out. I will retry the tool just to be sure (Attempt 2)...")
safe_refund_tool("TKT-999", 50)


--- The Agent Retries on a Network Timeout ---
[Agent] Calling refund tool (Attempt 1)...

[Tool Wrapper] Agent attempting refund of $50 for ticket TKT-999
💰 [Bank API] Successfully refunded $50.
[Agent] The server timed out. I will retry the tool just to be sure (Attempt 2)...

[Tool Wrapper] Agent attempting refund of $50 for ticket TKT-999
✅ [Safety Layer] BLOCKED: Refund for TKT-999 already processed. Preventing double-charge.


'ERROR: Refund already processed.'